In [2]:
import json
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

model = "claude-sonnet-4-5"

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

In [4]:
import html
import re
from concurrent.futures import ThreadPoolExecutor


def extract_json(text):
    """Pull the JSON out of Claude's reply, even if it adds code fences or extra words."""
    text = text.strip()
    fence = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    starts = [i for i in (text.find("["), text.find("{")) if i != -1]
    if not starts:
        raise ValueError("No JSON found in this reply:\n" + text)
    start = min(starts)
    end = max(text.rfind("]"), text.rfind("}"))
    return json.loads(text[start:end + 1])


class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def generate_dataset(self, task_description, prompt_inputs_spec,
                         num_cases=3, output_file="dataset.json"):
        spec_text = "\n".join(f'- "{k}": {v}' for k, v in prompt_inputs_spec.items())
        prompt = f"""Create {num_cases} varied test cases for evaluating a prompt.

The prompt's task: {task_description}

Each test case must have:
- "prompt_inputs": an object with exactly these keys:
{spec_text}
- "criteria": a list of 2 or 3 short statements describing what a good answer must contain for this specific case.

Return only a JSON array of {num_cases} objects. No other text."""
        messages = []
        add_user_message(messages, prompt)
        cases = extract_json(chat(messages))

        with open(output_file, "w") as f:
            json.dump({"task_description": task_description, "cases": cases}, f, indent=2)
        return cases

    def grade(self, task_description, case, output, extra_criteria):
        criteria = list(case.get("criteria", []))
        if extra_criteria:
            criteria.append(extra_criteria)
        criteria_text = "\n".join(f"- {c}" for c in criteria)
        prompt = f"""You are grading an answer produced by an AI prompt.

Task the prompt is meant to do: {task_description}

Inputs given to the prompt:
{json.dumps(case["prompt_inputs"], indent=2)}

Answer to grade:
<answer>
{output}
</answer>

The answer should meet these criteria:
{criteria_text}

Be strict. Return only a JSON object with these keys:
- "strengths": list of short strings
- "weaknesses": list of short strings
- "reasoning": one or two sentences
- "score": an integer from 1 to 10"""
        messages = []
        add_user_message(messages, prompt)
        verdict = extract_json(chat(messages))
        verdict["score"] = max(1, min(10, int(verdict["score"])))
        return verdict

    def run_evaluation(self, run_prompt_function, dataset_file,
                       extra_criteria=None, report_file="evaluation_report.html"):
        with open(dataset_file, "r") as f:
            data = json.load(f)
        if isinstance(data, dict):
            task_description = data.get("task_description", "")
            cases = data["cases"]
        else:
            task_description, cases = "", data

        def run_one(case):
            try:
                output = run_prompt_function(case["prompt_inputs"])
                verdict = self.grade(task_description, case, output, extra_criteria)
                return {"case": case, "output": output, **verdict}
            except Exception as e:
                return {"case": case, "output": "", "score": 0,
                        "strengths": [], "weaknesses": [],
                        "reasoning": f"This case failed to run: {e}"}

        with ThreadPoolExecutor(max_workers=self.max_concurrent_tasks) as pool:
            results = list(pool.map(run_one, cases))

        average = sum(r["score"] for r in results) / len(results)
        self.write_report(results, average, report_file)

        print(f"Average score: {average:.1f} / 10")
        for i, r in enumerate(results, 1):
            print(f"  Case {i}: {r['score']}/10 - {r['reasoning']}")
        print(f"Full report saved to {report_file}")
        return {"average_score": average, "results": results}

    def write_report(self, results, average, report_file):
        rows = ""
        for i, r in enumerate(results, 1):
            rows += (
                f"<h3>Case {i}: {r['score']}/10</h3>"
                f"<p><b>Inputs:</b> {html.escape(json.dumps(r['case']['prompt_inputs']))}</p>"
                f"<p><b>Grader:</b> {html.escape(str(r['reasoning']))}</p>"
                f"<p><b>Weaknesses:</b> {html.escape('; '.join(map(str, r['weaknesses'])))}</p>"
                f"<pre style='white-space:pre-wrap;background:#f4f4f4;padding:8px'>{html.escape(r['output'])}</pre><hr>"
            )
        page = f"<html><body style='font-family:sans-serif;max-width:800px;margin:auto'><h1>Average score: {average:.1f} / 10</h1>{rows}</body></html>"
        with open(report_file, "w", encoding="utf-8") as f:
            f.write(page)

In [14]:
def run_prompt(prompt_inputs):
    prompt = f"""
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts  
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [15]:
sample = {
    "height": "180 cm",
    "weight": "75 kg",
    "goal": "Build muscle",
    "restrictions": "Vegetarian",
}
print(run_prompt(sample))

C:\Users\andres.lazarte.it\AppData\Local\Temp\ipykernel_22808\313891637.py:20: DeprecationWarning: The model 'claude-sonnet-4-5' is deprecated and will reach end-of-life on November 30th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(**params)


# One-Day Athlete Meal Plan
**Goal: Muscle Building | Vegetarian**

## Daily Nutritional Targets
- **Total Calories:** 2,800 kcal
- **Protein:** 165g (24%)
- **Carbohydrates:** 360g (51%)
- **Fat:** 78g (25%)

---

## Meal 1 - Breakfast (7:00 AM)
**High-Protein Oatmeal Bowl**
- Rolled oats: 80g
- Whey protein powder (vanilla): 30g
- Banana: 120g
- Almond butter: 20g
- Chia seeds: 15g
- Blueberries: 80g
- Cinnamon: 2g

**Nutrition:** 650 kcal | 38g protein | 88g carbs | 18g fat

---

## Meal 2 - Mid-Morning Snack (10:00 AM)
**Greek Yogurt Parfait**
- Greek yogurt (non-fat): 200g
- Honey: 20g
- Walnuts: 20g
- Strawberries: 100g

**Nutrition:** 360 kcal | 24g protein | 38g carbs | 12g fat

---

## Meal 3 - Lunch (1:00 PM)
**Quinoa Power Bowl**
- Cooked quinoa: 200g
- Black beans (cooked): 150g
- Avocado: 80g
- Cherry tomatoes: 100g
- Spinach: 50g
- Feta cheese: 40g
- Olive oil: 10g
- Lemon juice: 15g

**Nutrition:** 720 kcal | 30g protein | 82g carbs | 28g fat

---

## Meal 4 - Pre-Workou

In [16]:
evaluator = PromptEvaluator(max_concurrent_tasks=3)

evaluator.generate_dataset(
    task_description="Write a compact, one-day meal plan for an athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    num_cases=3,
    output_file="meal_dataset.json",
)

C:\Users\andres.lazarte.it\AppData\Local\Temp\ipykernel_22808\313891637.py:20: DeprecationWarning: The model 'claude-sonnet-4-5' is deprecated and will reach end-of-life on November 30th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(**params)


[{'prompt_inputs': {'height': '175',
   'weight': '70',
   'goal': 'muscle gain',
   'restrictions': 'vegetarian'},
  'criteria': ['High protein vegetarian sources in each meal',
   'Caloric surplus appropriate for muscle building',
   'Specific meal timing around training']},
 {'prompt_inputs': {'height': '182',
   'weight': '85',
   'goal': 'weight loss',
   'restrictions': 'gluten-free, lactose intolerant'},
  'criteria': ['Caloric deficit while maintaining adequate protein',
   'No gluten or dairy products included',
   'Balanced macronutrients across all meals']},
 {'prompt_inputs': {'height': '168',
   'weight': '62',
   'goal': 'endurance performance',
   'restrictions': 'none'},
  'criteria': ['High carbohydrate content for sustained energy',
   'Hydration recommendations included',
   'Pre and post-workout nutrition specified']}]

In [17]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="meal_dataset.json",
    extra_criteria="""The answer should include: the daily calorie total, a macronutrient breakdown, exact foods with portions, and meal timing.""",
)

C:\Users\andres.lazarte.it\AppData\Local\Temp\ipykernel_22808\313891637.py:20: DeprecationWarning: The model 'claude-sonnet-4-5' is deprecated and will reach end-of-life on November 30th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(**params)


Average score: 7.7 / 10
  Case 1: 7/10 - The answer excellently meets all technical criteria with proper vegetarian protein distribution, caloric surplus, macro breakdown, portions, and timing. However, it fails the 'compact' requirement with excessive formatting and redundant information, and contains a confusing discrepancy between stated and actual calorie targets.
  Case 2: 9/10 - The answer excellently meets all specified criteria with comprehensive meal planning, proper dietary restrictions adherence, and detailed nutritional information. Minor weakness is lack of justification for the specific calorie target chosen for this athlete's profile and activity level.
  Case 3: 7/10 - The meal plan meets all required criteria with excellent detail on portions, timing, and nutrition. However, there are mathematical errors in the macronutrient percentages and the answer is incomplete/cut off.
Full report saved to evaluation_report.html
